<a href="https://colab.research.google.com/github/alb495/bettingModelAttempt/blob/F1/F1PowerRatings2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Set Up
1. Let's look at the 2025 season standings (the most recent data)
2. Let's see what the columns are actually named to avoid the KeyError
3. Create the Team Power Rankings

Based on current Ergast structure, it's often 'constructorNames' or 'constructorIds'
We will use the first constructor listed for each driver


In [1]:
import fastf1
from fastf1.ergast import Ergast
import pandas as pd

ergast = Ergast()

# 1.
standings = ergast.get_driver_standings(season=2025)
df_2025 = standings.content[0]

# 2.
print("Actual Columns in Data:", df_2025.columns.tolist())

# 3.
if 'constructorNames' in df_2025.columns:
    # This converts the list of constructors to a single string
    df_2025['team'] = df_2025['constructorNames'].apply(lambda x: x[0] if isinstance(x, list) else x)

    team_strength = df_2025.groupby('team')['points'].sum().sort_values(ascending=False)
    print("\n--- 2025 Team Power Rankings ---")
    print(team_strength)
else:
    print("\nColumn 'constructorNames' not found. Check the column list above!")

req         WARNING 	DEFAULT CACHE ENABLED! (32.0 KB) /root/.cache/fastf1


Actual Columns in Data: ['position', 'positionText', 'points', 'wins', 'driverId', 'driverNumber', 'driverCode', 'driverUrl', 'givenName', 'familyName', 'dateOfBirth', 'driverNationality', 'constructorIds', 'constructorUrls', 'constructorNames', 'constructorNationalities']

--- 2025 Team Power Rankings ---
team
McLaren           833.0
Mercedes          469.0
Red Bull          459.0
Ferrari           398.0
Williams          137.0
Aston Martin       89.0
RB F1 Team         84.0
Haas F1 Team       79.0
Sauber             70.0
Alpine F1 Team     22.0
Name: points, dtype: float64


# The 2026 Model

Fetching Data
1. Get all races for the season
2. We want a list of every driver's result in every race
  - a. Adding a 'round' number to help us weight by time
3. Combine into one big DataFrame

In [2]:
import fastf1.ergast
import pandas as pd
import numpy as np

ergast = fastf1.ergast.Ergast()
season = 2025

# 1.
races = ergast.get_race_results(season=season)

# 2.
all_results = []
for race in races.content:
    # 2a.
    race['round'] = race['number']
    all_results.append(race)

# 3.
df_season = pd.concat(all_results)

Weighting the Model

1. Define how "aggressive" the weight is.
0.9 means each previous race is worth 90% of the one after it.
2. Calculate the weight for each race
3. Calculate "Weighted Points"
4. Group by driver to see who finished the year strongest


In [3]:
# 1.
alpha = 0.9
max_round = df_season['round'].max()

# 2.
df_season['weight'] = alpha ** (max_round - df_season['round'])

# 3.
df_season['weighted_points'] = df_season['points'] * df_season['weight']

# 4.
weighted_standings = df_season.groupby(['givenName', 'familyName'])['weighted_points'].sum().sort_values(ascending=False)

print("--- Weighted Driver Form (End of 2025) ---")
print(weighted_standings.head(10))

--- Weighted Driver Form (End of 2025) ---
givenName    familyName
Oscar        Piastri       14.348907
Oliver       Bearman        4.000000
George       Russell        2.392993
Carlos       Sainz          0.034337
Esteban      Ocon           0.027389
Alexander    Albon          0.018864
Nico         Hülkenberg     0.010782
Lewis        Hamilton       0.010775
Andrea Kimi  Antonelli      0.007400
Lance        Stroll         0.006962
Name: weighted_points, dtype: float64


# 2026 Power Ratings
1. Pull live race results from Jolpica, the Ergast-compatible API
2. Use 2025 as the preseason baseline and blend in completed 2026 races as they happen
3. Reward recent form, podiums, wins, qualifying position, and constructor strength
4. Output a reusable ratings table for race-week predictions


In [ ]:
import pandas as pd
import numpy as np
from urllib.error import URLError
from urllib.request import urlopen
import json
import ssl

BASE_SEASON = 2025
TARGET_SEASON = 2026
API_BASE = "https://api.jolpi.ca/ergast/f1"


def fetch_json(url):
    try:
        with urlopen(url, timeout=20) as response:
            return json.loads(response.read().decode("utf-8"))
    except URLError as err:
        if isinstance(err.reason, ssl.SSLCertVerificationError):
            print("SSL certificate verification failed locally; retrying with an unverified context.")
            context = ssl._create_unverified_context()
            with urlopen(url, timeout=20, context=context) as response:
                return json.loads(response.read().decode("utf-8"))
        raise


def get_race_results(year):
    url = f"{API_BASE}/{year}/results.json?limit=2000"
    data = fetch_json(url)
    races = data["MRData"]["RaceTable"].get("Races", [])
    rows = []

    for race in races:
        for res in race.get("Results", []):
            driver = res["Driver"]
            rows.append({
                "season": int(year),
                "round": int(race["round"]),
                "race_name": race["raceName"],
                "date": race.get("date"),
                "driver_id": driver["driverId"],
                "driver": f"{driver.get('givenName', '')} {driver.get('familyName', '')}".strip(),
                "constructor": res["Constructor"]["name"],
                "grid": pd.to_numeric(res.get("grid"), errors="coerce"),
                "position": pd.to_numeric(res.get("positionOrder", res.get("position")), errors="coerce"),
                "points": float(res.get("points", 0)),
                "status": res.get("status"),
            })

    columns = [
        "season", "round", "race_name", "date", "driver_id", "driver",
        "constructor", "grid", "position", "points", "status"
    ]
    return pd.DataFrame(rows, columns=columns)


def get_race_schedule(year):
    url = f"{API_BASE}/{year}.json?limit=100"
    data = fetch_json(url)
    races = data["MRData"]["RaceTable"].get("Races", [])
    return pd.DataFrame([{
        "season": int(year),
        "round": int(race["round"]),
        "race_name": race["raceName"],
        "circuit": race["Circuit"]["circuitName"],
        "locality": race["Circuit"]["Location"].get("locality"),
        "country": race["Circuit"]["Location"].get("country"),
        "date": race.get("date"),
    } for race in races])


def score_results(results, alpha=0.92):
    if results.empty:
        return results.copy()

    scored = results.copy()
    scored["season_round"] = scored["season"] * 100 + scored["round"]
    max_round = scored["season_round"].max()
    scored["recency_weight"] = alpha ** (max_round - scored["season_round"])
    scored["finish_score"] = np.maximum(21 - scored["position"], 0).fillna(0)
    scored["grid_score"] = np.maximum(21 - scored["grid"], 0).fillna(0) * 0.25
    scored["podium_bonus"] = (scored["position"] <= 3).astype(int) * 4
    scored["win_bonus"] = (scored["position"] == 1).astype(int) * 5
    scored["rating_points"] = (
        scored["points"] + scored["finish_score"] + scored["grid_score"]
        + scored["podium_bonus"] + scored["win_bonus"]
    ) * scored["recency_weight"]
    return scored


base_results = get_race_results(BASE_SEASON)
current_results = get_race_results(TARGET_SEASON)
all_results = pd.concat([base_results, current_results], ignore_index=True)
scored_results = score_results(all_results)

if current_results.empty:
    print(f"No completed {TARGET_SEASON} races found yet. Ratings are using the {BASE_SEASON} baseline only.")
else:
    completed = current_results["round"].max()
    print(f"Loaded {len(current_results)} driver results from {TARGET_SEASON} through round {completed}.")

constructor_strength = (
    scored_results.groupby("constructor")["rating_points"]
    .sum()
    .sort_values(ascending=False)
)
constructor_multiplier = 1 + (constructor_strength / constructor_strength.max()).fillna(0) * 0.12

recent_form = (
    scored_results.sort_values(["season", "round"])
    .groupby("driver_id")
    .tail(5)
    .groupby("driver_id")
    .agg(recent_points=("rating_points", "sum"), recent_podiums=("podium_bonus", lambda s: (s > 0).sum()))
)

power_ratings = (
    scored_results.groupby(["driver_id", "driver"])
    .agg(
        constructor=("constructor", "last"),
        weighted_points=("rating_points", "sum"),
        avg_finish=("position", "mean"),
        starts=("race_name", "count"),
    )
    .join(recent_form, on="driver_id")
    .reset_index()
)

power_ratings["recent_points"] = power_ratings["recent_points"].fillna(0)
power_ratings["recent_podiums"] = power_ratings["recent_podiums"].fillna(0)
power_ratings["constructor_bonus"] = power_ratings["constructor"].map(constructor_multiplier).fillna(1.0)
power_ratings["power_rating"] = (
    power_ratings["weighted_points"] + power_ratings["recent_points"] * 0.35 + power_ratings["recent_podiums"] * 3
) * power_ratings["constructor_bonus"]
power_ratings["power_rank"] = power_ratings["power_rating"].rank(ascending=False, method="dense").astype(int)
power_ratings = power_ratings.sort_values("power_rating", ascending=False)

print("--- 2026 F1 Power Ratings ---")
print(power_ratings[["power_rank", "driver", "constructor", "power_rating", "avg_finish", "starts"]].head(20).to_string(index=False))


# Race Week Predictor
Pick a 2026 round, pull the schedule live, and rank likely finish order using the current power ratings. The default predicts the next scheduled race if results are already available, otherwise round 1.


In [ ]:
schedule_2026 = get_race_schedule(TARGET_SEASON)
completed_rounds = sorted(current_results["round"].unique().tolist()) if not current_results.empty else []
next_round = (max(completed_rounds) + 1) if completed_rounds else 1

# Change this number during race week.
RACE_ROUND = next_round

race_info = schedule_2026.loc[schedule_2026["round"] == RACE_ROUND]
if race_info.empty:
    raise ValueError(f"Round {RACE_ROUND} was not found in the {TARGET_SEASON} schedule.")

race = race_info.iloc[0]
race_week_prediction = power_ratings.copy()
race_week_prediction["race_round"] = RACE_ROUND
race_week_prediction["race_name"] = race["race_name"]
race_week_prediction["circuit"] = race["circuit"]
race_week_prediction["date"] = race["date"]
race_week_prediction["predicted_finish"] = range(1, len(race_week_prediction) + 1)

print(f"--- {TARGET_SEASON} Round {RACE_ROUND}: {race['race_name']} ---")
print(f"{race['circuit']} | {race['locality']}, {race['country']} | {race['date']}")
print(race_week_prediction[[
    "predicted_finish", "driver", "constructor", "power_rating", "avg_finish", "recent_points"
]].head(20).to_string(index=False))
